# TAC Python API Demo

Covers the full `tachiom.Tac` interface:
- **Run** Token-Aware Clustering on raw `.npy` files
- **Inspect** centroids and assignments
- **Analyse** the centroid budget allocation across token types
- **Save** centroids and assignments to disk
- **Feed** into `Tachiom.build_from_tac()` to build a full retrieval index

Input files (LOTTE, same as `tachiom_demo.ipynb`):

| File | Shape | dtype | Description |
|---|---|---|---|
| `documents.npy` | `[N, dim]` | `f16` | Token vectors |
| `document_token_ids_flat.npy` | `[N]` | `i64`/`u32` | Token-type ID per token |
| `doclens.npy` | `[n_docs]` | `i32`/`i64` | Tokens per document (only for Tachiom build) |

---
## 0b — Build the shared library

Run once per code change. `target-cpu=native` enables SIMD (AVX2/AVX-512) for PQ kernels.

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Find the project root (directory containing Cargo.toml) regardless of where
# the notebook is opened from.
def _find_project_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / "Cargo.toml").exists():
            return p
    raise RuntimeError("Could not find Cargo.toml — run this notebook from within the tachiom repo")

project_root = _find_project_root()

result = subprocess.run(
    ["maturin", "develop", "--release"],
    cwd=project_root,
    env={**os.environ, "RUSTFLAGS": "-C target-cpu=native"},
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("maturin build failed")
print("Build OK")

---
## 1 — Configuration

In [ ]:
from pathlib import Path

# ── Input paths — update these to match your local setup ─────────────────────
DATA_DIR = Path("/path/to/dataset")

VECTORS_FILE   = DATA_DIR / "documents.npy"
TOKEN_IDS_FILE = DATA_DIR / "document_token_ids_flat.npy"
DOCLENS_FILE   = DATA_DIR / "doclens.npy"   # only needed for Tachiom.build_from_tac()

# ── TAC output paths ──────────────────────────────────────────────────────────
TAC_DIR         = Path("/path/to/tac_output")
CENTROIDS_OUT   = TAC_DIR / "centroids.npy"     # [K, dim] f32
ASSIGNMENTS_OUT = TAC_DIR / "assignments.npy"   # [N] u32

# ── TAC params ────────────────────────────────────────────────────────────────
TAC_PARAMS = dict(
    n_centroids     = 2_097_152,
    n_iter          = 10,
    verbose         = True,
    max_sample_size = None,   # None = auto formula
)

print("Config OK")

---
## 2 — Run TAC

`Tac(n_centroids, *, n_iter, verbose, max_sample_size)` — configure once.  
`tac.train(vectors_path, token_ids_path)` — runs Token-Aware Clustering and populates `centroids`, `assignments`, etc.

In [ ]:
import time, tachiom

tac = tachiom.Tac(**TAC_PARAMS)
print(tac)   # not yet trained

t0 = time.perf_counter()
tac.train(str(VECTORS_FILE), str(TOKEN_IDS_FILE))
print(f"\nTAC done in {time.perf_counter() - t0:.2f}s")
print(tac)   # trained

---
## 3 — Inspect results

After `train()`, four properties are available: `n_centroids`, `dim`, `centroids` (f32), `centroids_f16` (f16), `assignments` (u32).

In [ ]:
import numpy as np

print(f"n_centroids : {tac.n_centroids:,}")
print(f"dim         : {tac.dim}")
print()
print(f"centroids      : shape={tac.centroids.shape}    dtype={tac.centroids.dtype}")
print(f"centroids_f16  : shape={tac.centroids_f16.shape}  dtype={tac.centroids_f16.dtype}")
print(f"assignments    : shape={tac.assignments.shape}  dtype={tac.assignments.dtype}")
print()

c = tac.centroids
print(f"centroid norms  — mean={np.linalg.norm(c, axis=1).mean():.4f}  "
      f"min={np.linalg.norm(c, axis=1).min():.4f}  max={np.linalg.norm(c, axis=1).max():.4f}")
print(f"assignments     — min={tac.assignments.min()}  max={tac.assignments.max()}  "
      f"(expected max = n_centroids - 1 = {tac.n_centroids - 1})")

In [ ]:
# f32 vs f16 round-trip error
c_f32 = tac.centroids
c_f16_as_f32 = tac.centroids_f16.astype(np.float32)

abs_diff = np.abs(c_f32 - c_f16_as_f32)
print("f32 vs f16 centroid values:")
print(f"  max absolute diff  : {abs_diff.max():.2e}")
print(f"  mean absolute diff : {abs_diff.mean():.2e}")

---
## 4 — Centroid budget analysis

TAC runs separate k-means per token type and distributes the budget with a damped-spread strategy:
- **Micro** tokens (< 128 occurrences) → 1 centroid each
- **Small** tokens (128–256 occurrences) → 2 centroids each
- **Active** tokens (≥ 256 occurrences) → damped score `√count × spread`, floor=4, budget-reconciled

Here we verify those properties by recovering the allocation from `assignments` + `token_ids`.

In [ ]:
# Load token IDs (same file that was passed to tac.train)
# Shape: [N]  dtype: i64  —  values are vocabulary IDs (BERT-style: 0..30521)
token_ids = np.load(TOKEN_IDS_FILE)
print(f"token_ids : shape={token_ids.shape}  dtype={token_ids.dtype}")
print(f"vocab size (unique token types) : {np.unique(token_ids).size:,}")
print(f"total tokens                    : {len(token_ids):,}")

In [ ]:
# For each unique token type: count how many tokens it has (frequency)
# and how many unique centroid IDs are assigned to it (allocated centroids).
#
# Strategy: sort by token_id once, then iterate over contiguous groups.
assignments = tac.assignments

sort_idx      = np.argsort(token_ids, kind='stable')
sorted_tids   = token_ids[sort_idx]
sorted_assigns = assignments[sort_idx]

unique_tids, tok_counts = np.unique(sorted_tids, return_counts=True)
split_points = np.cumsum(tok_counts)[:-1]
groups = np.split(sorted_assigns, split_points)

n_centroids_per_tid = np.array([np.unique(g).size for g in groups])

print(f"Unique token types : {len(unique_tids):,}")
print()

# ── Token frequency distribution ─────────────────────────────────────────────
micro = tok_counts < 128
small = (tok_counts >= 128) & (tok_counts < 256)
active = tok_counts >= 256
print(f"Micro  (< 128 occ)   : {micro.sum():5,} token types")
print(f"Small  (128-255 occ) : {small.sum():5,} token types")
print(f"Active (≥ 256 occ)   : {active.sum():5,} token types")
print()

# ── Centroid allocation distribution ─────────────────────────────────────────
print("Centroids per token type:")
print(f"  min    : {n_centroids_per_tid.min()}")
print(f"  max    : {n_centroids_per_tid.max():,}")
print(f"  mean   : {n_centroids_per_tid.mean():.1f}")
print(f"  median : {int(np.median(n_centroids_per_tid))}")
print(f"  total  : {n_centroids_per_tid.sum():,}  (should equal n_centroids={tac.n_centroids:,})")
print()

# Verify micro/small floors
assert (n_centroids_per_tid[micro] == 1).all(),  "micro tokens should have exactly 1 centroid"
assert (n_centroids_per_tid[small] == 2).all(),  "small tokens should have exactly 2 centroids"
assert (n_centroids_per_tid[active] >= 4).all(), "active tokens should have at least 4 centroids"
print("✓ Micro/small floors verified")

---
## 5 — Save to disk

Save `centroids` (f32) and `assignments` (u32) in NumPy format so they can be passed to `Tachiom.build_from_tac()` later — no need to re-run TAC.

In [ ]:
TAC_DIR.mkdir(parents=True, exist_ok=True)

np.save(CENTROIDS_OUT,   tac.centroids)     # f32  [K, dim]
np.save(ASSIGNMENTS_OUT, tac.assignments)   # u32  [N]

print(f"Saved centroids   → {CENTROIDS_OUT}   ({CENTROIDS_OUT.stat().st_size / 1e9:.2f} GB)")
print(f"Saved assignments → {ASSIGNMENTS_OUT}   ({ASSIGNMENTS_OUT.stat().st_size / 1e9:.2f} GB)")

# Verify round-trip
c_rt = np.load(CENTROIDS_OUT)
a_rt = np.load(ASSIGNMENTS_OUT)
assert np.array_equal(c_rt, tac.centroids),    "centroids round-trip mismatch"
assert np.array_equal(a_rt, tac.assignments),  "assignments round-trip mismatch"
print("✓ Round-trip verified")